MIN_CLUSTER_SIZE = 150   # clusters below this transcript count are auto-excluded
ALSO_EXCLUDE     = []    # list of global_id values to additionally exclude
EXCLUDE_FOV_STRIPS = [   # list of (fov, strip) tuples to exclude entirely
    # Example: (4, 'strip_2'),
]

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [ ]:
all_strips = pd.read_parquet('../data/processed/s1_all_strips_noise_flagged.parquet')
print(f'Loaded: {len(all_strips):,} transcripts')
print(f'Columns: {all_strips.columns.tolist()}')
print(f'\nPre-existing flags:')
print(f'  is_noise:         {all_strips["is_noise"].sum():,} ({100*all_strips["is_noise"].mean():.1f}%)')
print(f'  is_small_cluster: {all_strips["is_small_cluster"].sum():,} ({100*all_strips["is_small_cluster"].mean():.1f}%)')

In [ ]:
def build_cluster_table(df):
    """
    Build a summary table of all surviving DBSCAN clusters with globally unique IDs.

    Parameters
    ----------
    df : pd.DataFrame
        Noise-flagged transcript dataframe (output of nb08).

    Returns
    -------
    pd.DataFrame
        Columns: global_id, fov, strip, dbscan_label, n_transcripts.
        One row per unique surviving cluster across all FOVs and strips.
    """
    non_noise = df[~df['is_noise']]
    records = []
    gid = 0
    for (fov, strip, label), group in non_noise.groupby(['fov', 'strip', 'dbscan_label']):
        records.append({
            'global_id':    gid,
            'fov':          fov,
            'strip':        strip,
            'dbscan_label': int(label),
            'n_transcripts': len(group),
        })
        gid += 1
    return pd.DataFrame(records)

cluster_table = build_cluster_table(all_strips)
print(f'Total surviving clusters: {len(cluster_table)}')
print(cluster_table.to_string(index=False))

In [ ]:
cluster_table = build_cluster_table(df_flagged)
plot_cluster_overview(
    df_flagged,
    cluster_table,
    min_cluster_size=MIN_CLUSTER_SIZE,
    label_threshold=5000,
    exclude_fov_strips=EXCLUDE_FOV_STRIPS,
)

## Configuration
Edit the cell below, then re-run from here down.

- `MIN_CLUSTER_SIZE`: all clusters with fewer transcripts than this will be excluded.
- `ALSO_EXCLUDE`: list of `global_id` values from the table above to additionally exclude.

In [ ]:
MIN_CLUSTER_SIZE = 500
ALSO_EXCLUDE = []  # e.g. [3, 15] — add global_id values shown in the plot above

In [ ]:
df_cleaned = apply_cleanup(
    df_flagged,
    cluster_table,
    MIN_CLUSTER_SIZE,
    also_exclude=ALSO_EXCLUDE,
    exclude_fov_strips=EXCLUDE_FOV_STRIPS,
)

In [ ]:
# Preview: show what will be kept after cleanup
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
strips = ['strip_1', 'strip_2', 'strip_3']

for ax, strip_name in zip(axes, strips):
    s = cleaned[cleaned['strip'] == strip_name]
    kept    = s[~s['is_noise'] & ~s['manually_excluded']]
    removed = s[s['is_noise'] | s['manually_excluded']]

    ax.scatter(removed['x_global_px'], removed['y_global_px'],
               s=0.1, alpha=0.2, c='lightgrey', rasterized=True, label='excluded')
    ax.scatter(kept['x_global_px'], kept['y_global_px'],
               s=0.3, alpha=0.4, c='steelblue', rasterized=True, label='kept')
    ax.set_title(f'{strip_name}  |  kept={len(kept):,}  excluded={len(removed):,}')
    ax.set_aspect('equal')
    ax.legend(markerscale=8, fontsize=8)

fig.suptitle('Preview: transcripts after cleanup', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Total:             {len(cleaned):,}')
print(f'is_noise:          {cleaned["is_noise"].sum():,}')
print(f'is_small_cluster:  {cleaned["is_small_cluster"].sum():,}')
print(f'manually_excluded: {cleaned["manually_excluded"].sum():,}')
print(f'Kept (clean):      {(~cleaned["is_noise"] & ~cleaned["manually_excluded"]).sum():,}')

In [ ]:
cleaned.to_parquet('../data/processed/s1_all_strips_cleaned.parquet', index=False)
print('Saved: s1_all_strips_cleaned.parquet')
print(f'Columns: {cleaned.columns.tolist()}')